# TP 7 — La carte interactive du district d'Abidjan

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP7_Carte_interactive_Abidjan.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 7 · KOUADIO Kouao Laurent

Objectifs :
1. Calculer des indicateurs par sous-préfecture (bâti, arbres, eau,
   terres basses).
2. Faire une carte rapide avec `.explore()`.
3. Construire une carte **Folium** : fonds, choroplèthe, infobulles,
   couche raster, contrôle des couches.
4. Assembler un **tableau de bord** HTML.

Durée : environ 90 minutes. Connexion Internet nécessaire.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac rioxarray folium mapclassify

## 1. Imports

In [ ]:
import unicodedata
from pathlib import Path

import branca.colormap as bcm
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
import requests
import rioxarray
from folium.plugins import Fullscreen, MiniMap
from odc.stac import configure_rio, load
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.features import rasterize

configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10",
              GDAL_HTTP_RETRY_DELAY="2")
UTM = "EPSG:32630"

## 2. Les sous-préfectures du district

Même méthode qu'en séance 3 : copie locale, noms réparés, jointure
spatiale sur un point intérieur.

In [ ]:
def repare(t):
    try:
        t = t.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    return unicodedata.normalize("NFC", t)


def geoboundaries(niveau):
    fichier = Path(f"civ_{niveau.lower()}.geojson")
    if not fichier.exists():
        url = ("https://www.geoboundaries.org/api/current/gbOpen/"
               "CIV/" + niveau + "/")
        meta = requests.get(url, timeout=60).json()
        r = requests.get(meta["gjDownloadURL"], timeout=600)
        r.raise_for_status()
        fichier.write_bytes(r.content)
    g = gpd.read_file(fichier)
    g["shapeName"] = g["shapeName"].map(repare)
    return g


adm2, adm3 = geoboundaries("ADM2"), geoboundaries("ADM3")
district = adm2[adm2["shapeName"].str.contains("Abidjan")]
pts = adm3.copy()
pts["geometry"] = adm3.representative_point()
idx = gpd.sjoin(pts, district[["geometry"]], predicate="within").index
sp = (adm3.loc[idx, ["shapeName", "geometry"]]
      .rename(columns={"shapeName": "nom"}).reset_index(drop=True))
sp["km2"] = sp.to_crs(UTM).area / 1e6
bbox = [float(v) for v in district.total_bounds]
sp[["nom", "km2"]].round(1)

## 3. Occupation du sol par sous-préfecture (WorldCover)

On rasterise les sous-préfectures sur la grille WorldCover (10 m),
puis on compte les classes : c'est une **statistique zonale** (S3).

In [ ]:
url = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/"
       "2021/map/ESA_WorldCover_10m_2021_v200_N03W006_Map.tif")
wc = (rioxarray.open_rasterio(url).squeeze(drop=True)
      .rio.clip_box(*bbox, crs="EPSG:4326").load())
zones = rasterize(((g, i + 1) for i, g in enumerate(
    sp.to_crs(wc.rio.crs).geometry)), out_shape=wc.shape,
    transform=wc.rio.transform(), fill=0, dtype="int32")
df = pd.DataFrame({"z": zones.ravel(), "c": wc.values.ravel()})
parts = pd.crosstab(df.z, df.c, normalize="index") * 100
for code_wc, nom in {50: "bati", 10: "arbres", 80: "eau"}.items():
    sp[nom] = parts[code_wc].reindex(range(1, len(sp) + 1)).values
sp[["nom", "bati", "arbres", "eau"]].round(1)

**Question 1.** Quelle sous-préfecture est la plus bâtie ? La plus
boisée ? Pourquoi faut-il des **pourcentages** et non des km² pour
une choroplèthe ?

## 4. Une première carte en une ligne

In [ ]:
sp.explore(column="bati", cmap="Oranges", tooltip=["nom", "bati"],
           legend_kwds={"caption": "Bâti (%)"})

## 5. Les terres sous 5 m

Copernicus DEM est un **modèle de surface** : bâtiments et arbres
compris. On retire l'eau (WorldCover classe 80) pour ne garder que
les **terres**.

In [ ]:
cat = Client.open("https://earth-search.aws.element84.com/v1")
items = cat.search(collections=["cop-dem-glo-30"],
                   bbox=bbox).item_collection()
for it in items:
    a = it.assets["data"]
    a.href = a.href.replace("s3://copernicus-dem-30m/",
                            "https://copernicus-dem-30m.s3"
                            ".amazonaws.com/")
dem = load(items, bands=["data"], bbox=bbox, crs="EPSG:4326",
           resolution=0.0003).data.isel(time=0)
z = dem.values.astype(float)
eau = wc.rio.reproject_match(dem.rio.write_crs("EPSG:4326"),
                             resampling=Resampling.mode).values == 80
dedans = rasterize(((g, 1) for g in district.geometry),
                   out_shape=z.shape, transform=dem.odc.geobox.affine,
                   fill=0, dtype="uint8").astype(bool)
basses = (z < 5) & dedans & ~eau
print(f"terres sous 5 m : {100 * basses.sum() / (dedans & ~eau).sum():.1f} %"
      " des terres du district")

rgba = np.zeros(z.shape + (4,), dtype="uint8")
rgba[basses] = [33, 102, 172, 200]
b = dem.odc.geobox.boundingbox
limites = [[b.bottom, b.left], [b.top, b.right]]

**Question 2.** Recalculez la part **sans** retirer l'eau. De combien
change-t-elle ? Quelle valeur faut-il publier ?

## 6. La carte Folium

In [ ]:
m = folium.Map(tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
          "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics",
    name="Images Esri").add_to(m)

cmap = bcm.LinearColormap(["#fff5eb", "#fd8d3c", "#7f2704"],
                          vmin=0, vmax=float(sp["bati"].max()),
                          caption="Bâti (% de la surface)")
folium.GeoJson(
    sp.round(1), name="Bâti (%)",
    style_function=lambda f: {
        "fillColor": cmap(f["properties"]["bati"]),
        "color": "#1B4332", "weight": 1.5, "fillOpacity": 0.6},
    highlight_function=lambda f: {"weight": 3, "color": "#E07A1F"},
    tooltip=folium.GeoJsonTooltip(
        ["nom", "km2", "bati", "arbres"],
        aliases=["Sous-préfecture", "Surface (km²)", "Bâti (%)",
                 "Arbres (%)"])).add_to(m)
folium.raster_layers.ImageOverlay(
    rgba, bounds=limites, mercator_project=True,
    name="Terres sous 5 m", show=False).add_to(m)
cmap.add_to(m)
MiniMap(toggle_display=True).add_to(m)
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds(limites)
m.save("carte_abidjan.html")
m

**Question 3.** Activez la couche « Terres sous 5 m » sur le fond
« Images Esri ». Où se trouvent ces terres ? Une terre basse est-elle
une zone inondée ?

## 7. Le tableau de bord

In [ ]:
f1 = px.bar(sp.sort_values("bati"), x="bati", y="nom",
            orientation="h", height=380,
            labels={"bati": "bâti (%)", "nom": ""},
            color_discrete_sequence=["#E07A1F"],
            title="Part de bâti par sous-préfecture")
f2 = px.scatter(sp, x="bati", y="arbres", size="km2", text="nom",
                height=380, labels={"bati": "bâti (%)",
                                    "arbres": "arbres (%)"},
                color_discrete_sequence=["#1B4332"],
                title="Bâti et arbres")
f2.update_traces(textposition="top center")

carte = Path("carte_abidjan.html").read_text("utf-8")
carte = carte.replace('"', "&quot;")  # pour l'attribut srcdoc
kpi = {"Surface": f"{sp.km2.sum():.0f} km²",
       "Sous-préfectures": str(len(sp)),
       "Bâti moyen": f"{(sp.bati * sp.km2).sum() / sp.km2.sum():.1f} %"}
tuiles = "".join(f"<div class='kpi'><b>{v}</b><br>{k}</div>"
                 for k, v in kpi.items())
page = f"""<!doctype html><html lang="fr"><head><meta charset="utf-8">
<title>District d'Abidjan</title><style>
body{{font-family:Calibri,Arial;margin:16px;background:#EEF4F0}}
.kpis{{display:flex;gap:12px}} .kpi{{flex:1;background:white;
padding:10px;border-radius:8px}} .kpi b{{font-size:26px;
color:#E07A1F}} .grid{{display:grid;grid-template-columns:1.2fr 1fr;
gap:12px;margin-top:12px}} iframe{{width:100%;height:780px;border:0}}
</style></head><body><h2>District autonome d'Abidjan</h2>
<div class="kpis">{tuiles}</div><div class="grid">
<iframe srcdoc="{carte}"></iframe><div>
{f1.to_html(full_html=False, include_plotlyjs="cdn")}
{f2.to_html(full_html=False, include_plotlyjs=False)}
</div></div><p>Sources : ESA WorldCover 2021, Copernicus DEM,
geoBoundaries.</p></body></html>"""
Path("tableau_de_bord.html").write_text(page, "utf-8")
print("ouvrez tableau_de_bord.html dans un navigateur")

**Question 4.** Ajoutez un quatrième chiffre clé (par exemple la part
de terres sous 5 m). Que changeriez-vous pour un décideur municipal ?

**Question 5 (bonus).** Simplifiez les géométries avec
`sp.to_crs(UTM).simplify(50).to_crs(4326)` avant la carte. Comparez
la taille des fichiers HTML.

## À rendre

Le notebook exécuté, `carte_abidjan.html` et `tableau_de_bord.html`.